# MVPA L2 Results Review

This notebook summarizes the L2 MVPA analysis plan in `mvpa_L2.md`: pattern identification, SAD versus HC neural profiles, clinical relevance, SCR convergence, oxytocin modulation, and sensitivity analyses.

It is designed to run after `scripts/run_mvpa_l2_posthyak.sh` writes the post-Hyak outputs. Whole-brain/Schaefer sensitivity results are optional; if they are absent, the notebook skips them and reports which feature spaces are available.


In [ ]:
from pathlib import Path
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

root_env = os.environ.get('MVPA_L2_ROOT')
candidates = []
if root_env:
    candidates.append(Path(root_env))
candidates.extend([
    Path('/Users/xiaoqianxiao/projects/NARSAD/LSS/results/mvpa_l2'),
    Path('outputs/mvpa_l2'),
    Path('/output_dir/mvpa_l2'),
    Path('/gscratch/scrubbed/fanglab/xiaoqian/NARSAD/LSS/results/mvpa_l2'),
])
MVPA_ROOT = next((p for p in candidates if p.exists()), candidates[0])
HARMONIZED_DIR = MVPA_ROOT / 'harmonized'
STATS_DIR = MVPA_ROOT / 'stats'
print(f'MVPA_L2_ROOT = {MVPA_ROOT}')
print(f'Harmonized dir exists: {HARMONIZED_DIR.exists()}')
print(f'Stats dir exists: {STATS_DIR.exists()}')


In [ ]:
CORE_METRICS = [
    'Neural_Dist_Safety_Background',
    'Neural_ThreatLike_Safety',
    'Neural_SafetyLike_Safety',
    'Neural_Boundary_Separation',
    'Neural_Decision_Margin_CSS',
    'Neural_Safety_Trajectory_Slope',
    'Neural_Threat_Trajectory_Slope',
]

COMPANION_METRICS = [
    'Neural_Dist_Threat_Background',
    'Neural_Dist_Threat_Safety',
    'Neural_ThreatLike_Threat',
    'Neural_SafetyLike_Threat',
]

PRIMARY_CLINICAL = ['lsas_total', 'lsas_fear', 'lsas_avoid', 'dass_anxiety']
PRIMARY_SCR = ['SCR_SafetyMinusBackground', 'SCR_ThreatMinusSafety', 'SCR_Safety_Trajectory_Slope', 'SCR_Threat_Trajectory_Slope']

PATHS = {
    'subject_metrics': HARMONIZED_DIR / 'mvpa_l2_subject_metrics.csv',
    'scr_flags': HARMONIZED_DIR / 'scr_sensitivity_groups.csv',
    'aim2': STATS_DIR / 'aim2_group_difference.csv',
    'aim3': STATS_DIR / 'aim3_clinical_relevance.csv',
    'aim4': STATS_DIR / 'aim4_scr_convergence.csv',
    'aim5': STATS_DIR / 'aim5_oxytocin_modulation.csv',
    'primary_all': STATS_DIR / 'primary_models_all.csv',
    'sensitivity_all': STATS_DIR / 'sensitivity_models_all.csv',
    'summary_md': STATS_DIR / 'mvpa_l2_results_summary.md',
}

def read_csv_or_empty(path):
    path = Path(path)
    if not path.exists():
        print(f'Missing: {path}')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'Loaded {path.name}: {df.shape[0]} rows x {df.shape[1]} columns')
    return df

metrics_df = read_csv_or_empty(PATHS['subject_metrics'])
scr_flags_df = read_csv_or_empty(PATHS['scr_flags'])
aim2_df = read_csv_or_empty(PATHS['aim2'])
aim3_df = read_csv_or_empty(PATHS['aim3'])
aim4_df = read_csv_or_empty(PATHS['aim4'])
aim5_df = read_csv_or_empty(PATHS['aim5'])
primary_all_df = read_csv_or_empty(PATHS['primary_all'])
sensitivity_df = read_csv_or_empty(PATHS['sensitivity_all'])


## Data Availability

This section verifies which result tables are present and which feature spaces are available. The plan allows whole-brain/parcellation sensitivity to be absent.


In [ ]:
file_status = pd.DataFrame({'name': list(PATHS.keys()), 'path': [str(p) for p in PATHS.values()], 'exists': [Path(p).exists() for p in PATHS.values()]})
display(file_status)

if not metrics_df.empty and {'FeatureSpace', 'Group', 'Drug'}.issubset(metrics_df.columns):
    display(metrics_df.groupby(['FeatureSpace', 'Group', 'Drug'], dropna=False).size().rename('n').reset_index())
    available_feature_spaces = sorted(metrics_df['FeatureSpace'].dropna().unique())
else:
    available_feature_spaces = []
    print('Subject metric table is missing or does not contain FeatureSpace, Group, and Drug columns.')

print(f'Available feature spaces: {available_feature_spaces}')
if not any(str(fs).lower() in {'schaefer', 'wholebrain', 'wholebrain_schaefer'} for fs in available_feature_spaces):
    print('Whole-brain/Schaefer sensitivity results are not present; notebook will continue without them.')


In [ ]:
def select_columns(df, preferred):
    return [c for c in preferred if c in df.columns]

def tidy_result_table(df, n=30):
    if df.empty:
        return pd.DataFrame()
    cols = select_columns(df, ['analysis', 'sensitivity', 'feature_space', 'Group', 'metric', 'metric_z', 'clinical_score', 'clinical_score_z', 'scr_index', 'term', 'estimate', 'ci_low', 'ci_high', 'p', 'q', 'n', 'n_clinical_outliers_removed', 'n_metric_outliers_removed', 'r2', 'status'])
    out = df[cols].copy()
    if 'p' in out.columns:
        out['p_sort'] = pd.to_numeric(out['p'], errors='coerce')
        out = out.sort_values('p_sort', na_position='last').drop(columns='p_sort')
    return out.head(n)

def plot_forest(df, title, top_n=30):
    if df.empty or 'estimate' not in df.columns:
        print(f'No estimate table available for {title}.')
        return
    sub = df.copy()
    if 'status' in sub.columns:
        sub = sub[sub['status'].fillna('ok') == 'ok']
    needed = {'estimate', 'ci_low', 'ci_high'}
    if not needed.issubset(sub.columns):
        print(f'Missing estimate or confidence interval columns for {title}.')
        return
    label_cols = select_columns(sub, ['metric', 'clinical_score', 'scr_index', 'sensitivity', 'feature_space'])
    if not label_cols:
        label_cols = ['outcome'] if 'outcome' in sub.columns else []
    sub['label'] = sub[label_cols].astype(str).agg(' | '.join, axis=1) if label_cols else sub.index.astype(str)
    sub['p_sort'] = pd.to_numeric(sub.get('p', np.nan), errors='coerce')
    sub = sub.sort_values('p_sort', na_position='last').head(top_n)
    if sub.empty:
        print(f'No valid model rows for {title}.')
        return
    y = np.arange(len(sub))
    est = pd.to_numeric(sub['estimate'], errors='coerce')
    lo = pd.to_numeric(sub['ci_low'], errors='coerce')
    hi = pd.to_numeric(sub['ci_high'], errors='coerce')
    colors = ['tab:red' if q < 0.05 else 'tab:blue' for q in pd.to_numeric(sub.get('q', pd.Series(np.nan, index=sub.index)), errors='coerce').fillna(np.inf)]
    fig, ax = plt.subplots(figsize=(8, max(3, len(sub) * 0.28)))
    ax.axvline(0, color='0.5', linewidth=1)
    ax.errorbar(est, y, xerr=[est - lo, hi - est], fmt='none', ecolor='0.65', linewidth=1)
    ax.scatter(est, y, c=colors, s=30)
    ax.set_yticks(y)
    ax.set_yticklabels(sub['label'])
    ax.invert_yaxis()
    ax.set_xlabel('Model estimate with 95% CI')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

def signed_logp_heatmap(df, row_col, col_col, title):
    if df.empty or row_col not in df.columns or col_col not in df.columns:
        print(f'Cannot draw heatmap for {title}; required columns are absent.')
        return
    sub = df.copy()
    if 'status' in sub.columns:
        sub = sub[sub['status'].fillna('ok') == 'ok']
    if sub.empty or 'estimate' not in sub.columns or 'p' not in sub.columns:
        print(f'No valid rows for {title}.')
        return
    sub['signed_logp'] = np.sign(pd.to_numeric(sub['estimate'], errors='coerce')) * -np.log10(pd.to_numeric(sub['p'], errors='coerce').clip(lower=1e-300))
    mat = sub.pivot_table(index=row_col, columns=col_col, values='signed_logp', aggfunc='mean')
    if mat.empty:
        print(f'No heatmap values for {title}.')
        return
    fig, ax = plt.subplots(figsize=(max(5, mat.shape[1] * 1.4), max(3, mat.shape[0] * 0.35)))
    vmax = np.nanmax(np.abs(mat.values)) if np.isfinite(mat.values).any() else 1
    im = ax.imshow(mat.values, aspect='auto', cmap='coolwarm', vmin=-vmax, vmax=vmax)
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index)
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('signed -log10(p)')
    plt.tight_layout()
    plt.show()

def plot_metric_distribution(df, metric, feature_space='FearNetwork'):
    if df.empty or metric not in df.columns:
        print(f'Missing metric: {metric}')
        return
    required = {'FeatureSpace', 'Group', 'Drug'}
    if not required.issubset(df.columns):
        print(f'Missing columns for distribution plot: {required}')
        return
    sub = df[df['FeatureSpace'] == feature_space].copy()
    sub[metric] = pd.to_numeric(sub[metric], errors='coerce')
    sub = sub.dropna(subset=[metric])
    if sub.empty:
        print(f'No non-missing values for {metric} in {feature_space}.')
        return
    sub['cell'] = sub['Group'].astype(str) + '-' + sub['Drug'].astype(str)
    order = [x for x in ['HC-Placebo', 'SAD-Placebo', 'HC-Oxytocin', 'SAD-Oxytocin'] if x in set(sub['cell'])]
    if not order:
        order = sorted(sub['cell'].unique())
    fig, ax = plt.subplots(figsize=(max(5, len(order) * 1.25), 3.2))
    rng = np.random.default_rng(7)
    for i, cell in enumerate(order):
        vals = sub.loc[sub['cell'] == cell, metric].dropna().values
        if len(vals) == 0:
            continue
        jitter = rng.normal(0, 0.04, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, alpha=0.75, s=24)
        ax.hlines(np.nanmean(vals), i - 0.25, i + 0.25, color='black', linewidth=2)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(order, rotation=30, ha='right')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} in {feature_space}')
    plt.tight_layout()
    plt.show()


## Aim 1: Identify Vicarious Threat/Safety Patterns

Primary evidence: subject-aware cross-validated L2 logistic regression distinguishing `CSR` from `CSS`, with permutation testing. This notebook searches the harmonized table for decoding-related columns and displays available outputs. Feature importance/Haufe maps remain feature-space-specific outputs from the Hyak scripts.


In [ ]:
if metrics_df.empty:
    print('Subject metrics are not available yet.')
else:
    tokens = ['acc', 'auc', 'decode', 'decoding', 'cv', 'permutation', 'haufe', 'importance']
    exclude = {'Group', 'Drug', 'FeatureSpace', 'sub_ID'}
    decoding_cols = [c for c in metrics_df.columns if c not in exclude and any(t in c.lower() for t in tokens)]
    print(f'Found {len(decoding_cols)} candidate Aim 1 columns.')
    display(metrics_df[select_columns(metrics_df, ['sub_ID', 'FeatureSpace', 'Group', 'Drug'] + decoding_cols)].head(20))


## Aim 2: SAD Versus HC Neural Profile

Primary population: placebo SAD versus placebo HC in the primary FearNetwork feature space. Primary metrics include geometry, decoder evidence, boundary separation, and safety/threat trajectory metrics.


In [ ]:
display(tidy_result_table(aim2_df, n=50))
plot_forest(aim2_df, 'Aim 2: Placebo SAD vs HC neural profile', top_n=30)
if not metrics_df.empty:
    for metric in CORE_METRICS:
        plot_metric_distribution(metrics_df, metric, feature_space='FearNetwork')


## Aim 2 Learning-Dynamics Figure

This reproduces the trajectory-style figure from the stage-14 Hyak cache: safety movement toward `CS-`, threat maintenance toward reinstated `CSR`, and, when available, threat acquisition toward the shock/US target. The shock/US panel is optional and may be absent if shock target trials were not available in the feature-space cache.


In [ ]:
def payload_value(payload, *keys):
    if not isinstance(payload, dict):
        return None
    for key in keys:
        if key in payload:
            return payload[key]
    for value in payload.values():
        if isinstance(value, dict):
            found = payload_value(value, *keys)
            if found is not None:
                return found
    return None

def feature_space_dirs(feature_space='FearNetwork'):
    parent = MVPA_ROOT.parent if MVPA_ROOT.name == 'mvpa_l2' else MVPA_ROOT
    candidates = [
        parent / feature_space,
        MVPA_ROOT / feature_space,
        Path('/output_dir') / feature_space,
        Path('/gscratch/scrubbed/fanglab/xiaoqian/NARSAD/LSS/results') / feature_space,
        Path('/Users/xiaoqianxiao/projects/NARSAD/LSS/results') / feature_space,
        Path('outputs/mvpa_l2') / feature_space,
    ]
    return [p for p in candidates if p.exists()]

def find_trajectory_payload(feature_space='FearNetwork'):
    names = [
        'cell_14.joblib',
        'checkpoints/cell_14.joblib',
        'cell_12_trajectories.joblib',
        'checkpoints/cell_12_trajectories.joblib',
        'intermediate/stage14_trajectories.joblib',
        'stage14_trajectories.joblib',
    ]
    try:
        import joblib
    except ImportError:
        print('joblib is not available in this kernel; cannot load trajectory cache.')
        return None, None
    for base in feature_space_dirs(feature_space):
        for name in names:
            path = base / name
            if path.exists():
                payload = joblib.load(path)
                results = payload_value(payload, 'results_13_2') or payload
                if isinstance(results, dict):
                    return results, path
    return None, None

def summarize_trajectory(df):
    if df is None or df.empty:
        return pd.DataFrame()
    needed = {'trial', 'score', 'Group'}
    if not needed.issubset(df.columns):
        print(f'Trajectory dataframe missing columns: {needed - set(df.columns)}')
        return pd.DataFrame()
    sub = df.copy()
    sub['trial'] = pd.to_numeric(sub['trial'], errors='coerce')
    sub['score'] = pd.to_numeric(sub['score'], errors='coerce')
    sub = sub.dropna(subset=['trial', 'score', 'Group'])
    summary = sub.groupby(['Group', 'trial'], dropna=False)['score'].agg(['mean', 'sem', 'count']).reset_index()
    summary['sem'] = summary['sem'].fillna(0)
    return summary

def annotate_trial_p(ax, stats_df, y=1.68):
    if stats_df is None or stats_df.empty:
        return
    trial_col = next((c for c in ['trial', 'Trial', 'block', 'Block'] if c in stats_df.columns), None)
    p_col = next((c for c in ['p', 'p_value', 'pval', 'P', 'p_unc', 'p_group'] if c in stats_df.columns), None)
    if trial_col is None or p_col is None:
        return
    temp = stats_df.copy()
    temp[trial_col] = pd.to_numeric(temp[trial_col], errors='coerce')
    temp[p_col] = pd.to_numeric(temp[p_col], errors='coerce')
    temp = temp.dropna(subset=[trial_col, p_col])
    for _, row in temp.iterrows():
        if row[p_col] < 0.05:
            x = row[trial_col]
            ax.plot([x - 0.18, x + 0.18], [y, y], color='black', linewidth=1)
            ax.text(x, y + 0.03, f'p={row[p_col]:.3f}'.replace('0.', '.'), ha='center', va='bottom', fontsize=8)

def plot_trajectory_panel(ax, df, stats_df, title, target_label, target_color, marker):
    summary = summarize_trajectory(df)
    if summary.empty:
        ax.text(0.5, 0.5, 'Not available', transform=ax.transAxes, ha='center', va='center')
        ax.set_title(title)
        ax.set_axis_off()
        return
    colors = {'SAD': '#c44e52', 'HC': '#4c72b0'}
    for group in ['SAD', 'HC']:
        cur = summary[summary['Group'] == group].sort_values('trial')
        if cur.empty:
            continue
        x = cur['trial'].to_numpy(dtype=float)
        y = cur['mean'].to_numpy(dtype=float)
        sem = cur['sem'].to_numpy(dtype=float)
        ax.plot(x, y, marker=marker, linewidth=2, color=colors.get(group, '0.25'), label=group)
        ax.fill_between(x, y - sem, y + sem, color=colors.get(group, '0.25'), alpha=0.18)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, label='Start')
    ax.axhline(1, color=target_color, linestyle='-', linewidth=1.5, label=target_label)
    annotate_trial_p(ax, stats_df)
    ax.set_title(title)
    ax.set_xlabel('Trial (Block Size: 1)')
    ax.set_ylim(-1.0, 2.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)

trajectory_results, trajectory_path = find_trajectory_payload('FearNetwork')
if trajectory_results is None:
    print('No FearNetwork stage-14 trajectory cache found. Expected cell_14.joblib or cell_12_trajectories.joblib in the FearNetwork output directory.')
else:
    print(f'Loaded trajectory cache: {trajectory_path}')
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.2), sharey=True)
    plot_trajectory_panel(axes[0], trajectory_results.get('data_safe', pd.DataFrame()), trajectory_results.get('stats_safe', pd.DataFrame()), 'A. Safety Trajectory\n(Target = CS-)', 'Target (CS-)', '#2ca02c', 'o')
    axes[0].set_ylabel('Similarity Score (0=Start, 1=Target)')
    plot_trajectory_panel(axes[1], trajectory_results.get('data_threat', pd.DataFrame()), trajectory_results.get('stats_threat', pd.DataFrame()), 'B. Threat Maintenance\n(Target = Reinstated CSR)', 'Target (Reinstated CSR)', '#d62728', 's')
    plot_trajectory_panel(axes[2], trajectory_results.get('data_threat_shock', pd.DataFrame()), trajectory_results.get('stats_threat_shock', pd.DataFrame()), 'C. Threat Acquisition\n(Target = Shock/US)', 'Target (Shock/US)', '#9467bd', '^')
    plt.tight_layout()
    out_path = STATS_DIR / 'mvpa_l2_learning_dynamics_trajectory.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f'Saved trajectory figure -> {out_path}')
    plt.show()


## Aim 2 Companion Metrics

Companion metrics clarify whether observed differences are safety-specific, threat-specific, or broader reorganization of threat-safety neural space. These are interpretive rather than the main primary family.


In [ ]:
if metrics_df.empty:
    print('Subject metrics are not available yet.')
else:
    companion_available = [m for m in COMPANION_METRICS if m in metrics_df.columns]
    print(f'Available companion metrics: {companion_available}')
    for metric in companion_available:
        plot_metric_distribution(metrics_df, metric, feature_space='FearNetwork')


## Aim 3: Clinical Relevance

Primary question: within each diagnostic group, do z-scored neural metrics relate to z-scored LSAS and DASS anxiety measures after applying the FearNetwork stage-29 z-score outlier rule, while adjusting for available covariates? Interpret dimensional anxiety associations carefully if they are driven only by group separation.


In [ ]:
display(tidy_result_table(aim3_df, n=80))
signed_logp_heatmap(aim3_df, 'metric', 'clinical_score', 'Aim 3: signed clinical association strength')
plot_forest(aim3_df, 'Aim 3: strongest clinical associations', top_n=30)


## Aim 4: Physiological Convergence With SCR

Primary question: do neural metrics align with SCR indices of safety/threat learning? SCR responder/learner cohorts are sensitivity populations, not replacements for the full SCR-fMRI sample.


In [ ]:
display(tidy_result_table(aim4_df, n=80))
signed_logp_heatmap(aim4_df, 'metric', 'scr_index', 'Aim 4: signed SCR-neural convergence strength')
plot_forest(aim4_df, 'Aim 4: strongest SCR convergence tests', top_n=30)
if not scr_flags_df.empty:
    flag_cols = [c for c in scr_flags_df.columns if c.startswith('SCR_')]
    counts = scr_flags_df[flag_cols].sum().rename('n').reset_index().rename(columns={'index': 'SCR cohort'}) if flag_cols else pd.DataFrame()
    display(counts)


## Aim 5: Oxytocin Modulation

Primary model: `neural_metric ~ Group * Drug + covariates`. Interpret direction and uncertainty, not only p-values. Movement of SAD-oxytocin toward HC-placebo can be described as an HC-reference shift, not automatically as improvement.


In [ ]:
display(tidy_result_table(aim5_df, n=80))
plot_forest(aim5_df, 'Aim 5: Group x Drug modulation', top_n=30)
if not metrics_df.empty:
    for metric in CORE_METRICS:
        plot_metric_distribution(metrics_df, metric, feature_space='FearNetwork')


## Sensitivity Analyses

Sensitivity tests include alternative masks or feature spaces and SCR responder/learner cohorts. Whole-brain/Schaefer sensitivity is optional and may be absent.


In [ ]:
if sensitivity_df.empty:
    print('No sensitivity model table found yet.')
else:
    display(tidy_result_table(sensitivity_df, n=100))
    if 'sensitivity' in sensitivity_df.columns:
        display(sensitivity_df.groupby(['analysis', 'sensitivity'], dropna=False).size().rename('n_tests').reset_index())
    if 'feature_space' in sensitivity_df.columns:
        print('Sensitivity feature spaces:', sorted(sensitivity_df['feature_space'].dropna().unique()))
    has_wholebrain = False
    if 'feature_space' in sensitivity_df.columns:
        has_wholebrain = any(str(x).lower() in {'schaefer', 'wholebrain', 'wholebrain_schaefer'} for x in sensitivity_df['feature_space'].dropna().unique())
    if not has_wholebrain:
        print('Whole-brain/Schaefer sensitivity was not run or not available. This is allowed by the current workflow.')
    plot_forest(sensitivity_df, 'Sensitivity analyses: strongest effects', top_n=40)


## Integrated Primary Result Summary

This section combines primary model tables and highlights rows that survive FDR when available. Use it as a triage view before writing the results narrative.


In [ ]:
if primary_all_df.empty:
    frames = [df for df in [aim2_df, aim3_df, aim4_df, aim5_df] if not df.empty]
    primary_view = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
else:
    primary_view = primary_all_df.copy()

if primary_view.empty:
    print('No primary model results available yet.')
else:
    display(tidy_result_table(primary_view, n=120))
    if 'q' in primary_view.columns:
        q = pd.to_numeric(primary_view['q'], errors='coerce')
        sig = primary_view[q < 0.05].copy()
        print(f'FDR-significant rows: {len(sig)}')
        display(tidy_result_table(sig, n=120))
    if 'p' in primary_view.columns:
        p = pd.to_numeric(primary_view['p'], errors='coerce')
        suggestive = primary_view[(p < 0.05) & ~(pd.to_numeric(primary_view.get('q', np.nan), errors='coerce') < 0.05)].copy()
        print(f'Nominal p<.05 but not FDR-significant rows: {len(suggestive)}')
        display(tidy_result_table(suggestive, n=80))


## Saved Markdown Summary

If `scripts/summarize_mvpa_l2_results.py` has run, the compact Markdown summary is shown below.


In [ ]:
summary_path = PATHS['summary_md']
if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print(f'Missing summary file: {summary_path}')


## Reporting Checklist

Before manuscript reporting, confirm: subject counts by `Group * Drug`, available feature spaces, missingness for each primary metric, model covariates, FDR families, and whether sensitivity results are consistent with the primary FearNetwork results. Report absent whole-brain/Schaefer sensitivity as pending or not performed, rather than treating it as a negative result.
